# Imports

In [ ]:
import os
import glob
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
import math
from functools import partial
import numpy as np
import scipy as sp
import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from functools import partial
import datetime
import json
from PIL import Image
from attr import dataclass
import scipy.io as scio
import h5py

import torch
from torch import nn, Tensor, optim
import torch.nn.functional as F
from torchvision import datasets
from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure, StructuralSimilarityIndexMeasure
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, StepLR, SequentialLR, LinearLR, OneCycleLR
from torch.utils.data import  DataLoader
from torchvision.transforms import v2
from torchinfo import summary

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping, TQDMProgressBar
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pynop

from utils import plot_interactive, plot_comparison, get_simulation_by_id, animate_results_mpl

%matplotlib inline


# Model creation


In [ ]:
# Dictionary containing all model configurations
# number of past slice (ground truth) given to the network or latent representation the case of CausalLNO
steps = 6

models_config = {
    "Galerkin": {
        "model": pynop.GalerkinTransformer,
        "params": {
            "in_channels": 2 * steps,
            "out_channels": 2,
            "dt": 5 / 100,
            "num_blocks": 4,
            "hidden_channels": 128,
            "num_heads": 4,
            "mlp_layers": 2,
            "mlp_dim": 256,
            "activation": nn.GELU,
            "mlp_factor": 2,
            "dim": 2,
            "verbose": False,
        },
    },
    "ITLNO": {
        "model": pynop.ITLNO,
        "params": {
            "in_channels": 2 * steps,
            "out_channels": 2,
            "modes": 16,
            "dt": 5 / 100,
            "num_blocks": 4,
            "hidden_channels": 128,
            "num_heads": 4,
            "linear_kernel": False,
            "mlp_layers": 1,
            "mlp_dim": 128,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "mlp_act": nn.GELU,
            "dim": 2,
            "orthogonal_init": True,
            "pe": True,
            "rmsnorm": True,
            "basis_mode": "free",
            "verbose": True,
        },
    },
    "chebITLNO": {
        "model": pynop.ITLNO,
        "params": {
            "in_channels": 2 * steps,
            "out_channels": 2,
            "dt": 5.0 / 100.0,  # final t=5, 101 slices
            "modes": 16,
            "num_blocks": 4,
            "hidden_channels": 128,
            "num_heads": 4,
            "linear_kernel": False,
            "mlp_layers": 1,
            "mlp_dim": 128,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "mlp_act": nn.GELU,
            "dim": 2,
            "orthogonal_init": True,
            "pe": True,
            "basis_mode": "free",
            "rmsnorm": True,
            "verbose": True,
        },
    },
    "LatentGalerkin": {
        "model": pynop.LatentGalerkin,
        "params": {
            "in_channels": 2 * steps,
            "out_channels": 2,
            "modes": 128,
            "dt": 5 / 100,
            "num_blocks": 4,
            "hidden_channels": 128,
            "num_heads": 4,
            "mlp_layers": 1,
            "mlp_dim": 128,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "mlp_act": nn.GELU,
            "dim": 2,
            "latent_attention": "galerkin",
            "verbose": True,
        },
    },
    "Transolver_classic": {
        "model": pynop.Transolver,
        "params": {
            "in_ch": 2 * steps,
            "out_ch": 2,
            "slice_num": 64,
            "n_layers": 8,
            "n_hidden": 128,
            "dropout": 0,
            "n_head": 4,
            "activation": nn.GELU,
            "mlp_ratio": 1,
            "dim": 2,
            "cond_dim": None,
        },
    },
    "Transolver_linear": {
        "model": pynop.Transolver,
        "params": {
            "in_ch": 2 * steps,
            "out_ch": 2,
            "slice_num": 64,
            "n_layers": 8,
            "n_hidden": 128,
            "dropout": 0.05,
            "n_head": 4,
            "mode": "linear",
            "activation": nn.GELU,
            "mlp_ratio": 2,
            "dim": 2,
            "cond_dim": None,
        },
    },
    "LatentTransolver3": {
        "model": pynop.LatentTransolver,
        "params": {
            "in_channels": 2 * steps,
            "out_channels": 2,
            "dt": 5 / 100,
            "modes": 64,
            "num_blocks": 4,
            "hidden_channels": 128,
            "dropout": 0.05,
            "num_heads": 4,
            "activation": nn.GELU,
            "mlp_act": nn.GELU,
            "mlp_factor": 2,
            "dim": 2,
            "cond_dim": None,
            "rmsnorm": True,
            "std_init": 1e-3,
            "verbose": True,
        },
    },
    "FNO": {
        "model": pynop.FNO,
        "params": {
            "in_channels": 2 * steps,
            "out_channels": 2,
            "modes": (16, 16),
            "hidden_channels": (36, 36, 36, 36),
            "blocks": ["FNO", "FNO", "FNO", "FNO"],
            "spectral_compression_factor": (1, 1, 1),
            "trainable_pos_encoding": False,
            "fixed_pos_encoding": True,
            "trainable_pos_encoding_modes": (16, 16),
            "trainable_pos_encoding_dims": 8,
        },
    },"TimeAttentionLatentGalerkin": {
        "model": pynop.TimeAttentionLatentGalerkin,
        "params": {
            "in_channels": 2,
            "out_channels": 2,
            "modes": 128,
            "dt": 5 / 100,
            "num_blocks": 4,
            "hidden_channels": 128,
            "num_heads": 4,
            "mlp_layers": 1,
            "mlp_dim": 128,
            "mlp_factor": 2,
            "activation": nn.GELU,
            "mlp_act": nn.GELU,
            "dim": 2,
            "latent_attention": "galerkin",
            "verbose": True,
            "memory": steps,
        },
    },
}

# name = "ITLNO"
# name = "chebITLNO"
# name = 'Transolver_classic'
# name = "Transolver_linear"
# name = 'FNO'
# name = "LatentTransolver3"
# name = "Galerkin"
# name = "LatentGalerkin"
name = "TimeAttentionLatentGalerkin"

# Instantiate model
if name == "CLNO" or name == "TimeAttentionLatentGalerkin":
    in_steps = 1
else:
    in_steps = steps

model = models_config[name]["model"](**models_config[name]["params"]).to("cuda")

In [ ]:
# summary(model, input_size=(1,2,128,128))
data = torch.rand((2, 2*in_steps, 128, 128)).to("cuda")
time = torch.rand(2).unsqueeze(-1).float().to("cuda")
if name == "FNO":
    print(summary(model, input_data=data, depth=4, col_names=("input_size", "output_size", "num_params")))
elif name == "TimeAttentionLatentGalerkin":
    _, history = model(x=data, time=time)
    print(summary(model, input_data=(data, time, history), depth=4, col_names=("input_size", "output_size", "num_params")))
else:
    print(summary(model, input_data=(data, time), depth=4, col_names=("input_size", "output_size", "num_params")))


# Data loading, creation of the train/val sets and dataloaders

In [ ]:
# datapath = Path("/media/jlux/SSD2/pdebench/2d_reaction_diffusion/133017.hdf5")
# stats=pynop.inspect_PDEBenchDataset(datapath, DT=5e-2)
# print(stats)
# With DT=5/100²
# ==================================================
#               PDEBENCH INSPECTION REPORT
# ==================================================
# Simulations : 1000
# Time steps  : 101
# Resolution  : (128, 128)
# Channels    : 2
# --------------------------------------------------
# CHANNEL 0:
#   [Values] Min: -5.2797 | Max: 5.5866
#   [Values] Mean: -0.0311 | Std: 0.1436
#   [Deriv ] Mean: -0.014908 | Std: 1.962596
# --------------------------------------------------
# CHANNEL 1:
#   [Values] Min: -5.3747 | Max: 5.1440
#   [Values] Mean: -0.0199 | Std: 0.1113
#   [Deriv ] Mean: -0.011219 | Std: 1.992754
# --------------------------------------------------
# ==================================================

In [ ]:
# datapath = Path("F:/Projets/2D_diff-react_NA_NA.h5")
datapath = Path("/media/jlux/SSD2/pdebench/2d_reaction_diffusion/133017.hdf5")

# stats du dataset
# 'global_min': array([-5.279711 , -5.3747244], dtype=float32),
# 'global_max': array([5.5866303, 5.1440244], dtype=float32)

# shift = 5.59
# scale = 0.5
std = (0.1113 + 0.1436) * 0.5
shift = 0  # -> values > 0
scale = 1 #/ std  # Division by the std to get std=1

unrolling = 10

train_set = pynop.PDEBenchDataSet(
    datapath,
    T_unroll=unrolling + steps,
    step=8,
    load_in_ram=True,
    split_type="train",
    split_ratio=0.9,
    n_samples=100,
    seed=42,
    min_id=2,
    max_id=70,
    shift=shift,
    scale=scale,
)
val_set = pynop.PDEBenchDataSet(
    datapath,
    T_unroll=unrolling + steps,
    step=8,
    load_in_ram=True,
    split_type="val",
    split_ratio=0.9,
    n_samples=100,
    seed=42,
    min_id=2,
    max_id=70,
    shift=shift,
    scale=scale,
)
print(len(train_set))

In [ ]:
batch_size = 10
train_dataloader = DataLoader(train_set, shuffle=True, batch_size=batch_size, num_workers=10)
valid_dataloader = DataLoader(
    val_set, shuffle=False, batch_size=batch_size, num_workers=10, persistent_workers=True, pin_memory=True
)

# Preparing the training
**Notes:**

- It seems better to use the class `pynop.nL2Loss` or `pynop.nMSELoss` for the loss. It computes the normalized MSE or RMSE *by canal* (i.e. for each physical variable), especially if the variables have different order of magnitude.

- The metric RMSE, which is logged during training, is a mean over all spatial, channel and batch dimension. It is used here as it is often the metric given in the papers.

- Most of the  benchmarks use at least 10 previous time steps as input to predict the next time step.
- In Transolver, LNO and LinearNO, they mostly use Onecycle scheduler without curriculum training (full 10 timesteps rollout  at first epoch). 

## Freeze layers

In [ ]:
# for named_param, param in model.named_parameters():
#     # Only allow gradients for the temporal block
#     # OR if it's a normalization layer (optional strategy)
#     if "temporal_block" in named_param or "norm" in named_param:
#         param.requires_grad = True
#     else:
#         param.requires_grad = False

# print("Trainable parameters:")
# for named_param, param in model.named_parameters():
#     if param.requires_grad:
#         print(f"{named_param}")

In [ ]:
from pynop import ITLNOGradientLogger
exp = "fullds"  # root directory
weight_decay = 1e-4
max_epochs = 250
lr = 1e-3  # This is the maximum learning rate
start_monitoring = 70  # start saving the best models only after this epoch
mse_shift = 0
mse_scale = 1
residual = False

# pynop.nDWMSELoss(shift=mse_shift, scale=mse_scale),
# pynop.SpectralLoss( beta=1),
# pynop.SobolevLoss(l1=1., l2=1.),
# pynop.nMSELoss(),
# torch.nn.MSELoss(),
# nn.L1Loss() better for autoencoder

def SSMLoss(preds, targets):
    SSM = StructuralSimilarityIndexMeasure().to("cuda")
    return 1 - SSM(preds, targets)

print("steps", steps)

train_config = pynop.TrainingConfig(
    start_autoregressive=0,  # epoch at which the AR steps starts increasing
    final_autoregressive=80,
    min_autoregressive_steps=4,  # number of AR steps before start_autoregressive
    max_autoregressive_steps=unrolling,
    detach_grad_steps=40,
    derivative_loss_weight=1,
    loss_fn=[pynop.DWMSELoss(), pynop.SpectralLoss(beta=2)],
    loss_weights=[1000, 1000],
    noise_level=std*1e-2,
    time_normalization=101,
    n_slices=steps,  # number of previous time steps used to predict the next one
    temporal_weighting=1.05,
    train_mode="predict",  # only useful for Causal NO Models: if train_mode=='autoencoder', the Temporalmodule is disabled
)

scheduler_config = [
    {
        "scheduler": OneCycleLR,
        "max_lr": lr,
        "div_factor": 1e3,
        "final_div_factor": 1e3,
        "pct_start": 0.05,
        "interval": "step",
        "total_steps": max_epochs * len(train_dataloader),
        "frequency": 1,
    },
]



print("Total number of steps:", max_epochs * len(train_dataloader))
# model_name = model.__class__.__name__
model_name = name
loss_name = str(type(train_config.loss_fn)).split(".")[-1]
folder = f"{model_name}"  # _{loss_name}"
folder = Path(folder)
now = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
baselogdir = Path("/media/jlux/SSD2/ReactionDiffusion/") / Path(exp) / folder
logdir = Path(baselogdir / Path(f"{steps}steps_{now}"))
# logdir = Path("/media/jlux/SSD2/ReactionDiffusion/fullds/ITLNO_res/20260223-202651")
callbacks = []

loggers = [TensorBoardLogger(logdir / Path("tb_logs")), CSVLogger(logdir)]

callbacks.append(
    ModelCheckpoint(
        monitor="val_loss", filename=os.path.join(logdir, "best_val_loss"), mode="min", save_top_k=2, save_last=False
    )
)

callbacks.append(
    pynop.CurriculumCheckpoint(
        start_epoch=start_monitoring,
        monitor="loss",
        filename=os.path.join(logdir, "best_train_loss"),
        mode="min",
        save_top_k=2,
        save_last=False,
    )
)
callbacks.append(
    pynop.CurriculumCheckpoint(
        start_epoch=start_monitoring,
        dirpath=logdir,
        filename="last",
        every_n_epochs=1,
        save_top_k=1,
        mode="max",
    )
)

callbacks.append(LearningRateMonitor(logging_interval="epoch"))

# print values in scientific format
class CustomProgressBar(TQDMProgressBar):
    def get_metrics(self, trainer, model):
        items = super().get_metrics(trainer, model)
        # On applique le format scientifique à tout le dictionnaire
        return {k: (f"{v:.3e}" if isinstance(v, float) else v) for k, v in items.items()}

callbacks.append(CustomProgressBar(leave=True))

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad and p.ndim != 2, model.parameters()),
    lr=lr,
    weight_decay=weight_decay,
    betas=(0.9, 0.99),
)



if name =="TimeAttentionLatentGalerkin":
    lightning_model = pynop.CausalNOModel(
        model=model, train_config=train_config, optimizer=optimizer, scheduler_config=scheduler_config
    )
else:
    lightning_model = pynop.MultiStepNOModel(
        model=model, train_config=train_config, optimizer=optimizer, scheduler_config=scheduler_config
    )

# --------------------- Load checkpoint here ---------------------
# checkpoint = torch.load(Path("/media/jlux/SSD2/ReactionDiffusion/fullds/LatentGalerkin/8steps_20260320-150930") / Path("last.ckpt"))
# lightning_model.load_state_dict(checkpoint['state_dict'])
# --------------------- load the optimizer checkpoint to continue a previous run ---------------------
# optimizer.load_state_dict(checkpoint['optimizer_states'][0])

torch.set_float32_matmul_precision("highest")
# torch.set_float32_matmul_precision("high")

trainer = pl.Trainer(
    max_epochs=max_epochs,
    # limit_train_batches=0.5,
    callbacks=callbacks,
    accelerator="gpu",
    logger=loggers,
    num_sanity_val_steps=0,
    log_every_n_steps=50,
    gradient_clip_val=1,
    gradient_clip_algorithm="norm",
)

print("Log dir:", logdir)

# Tensorboard

In [ ]:
%load_ext tensorboard
# %tensorboard --logdir $baselogdir


# Training the model

In [ ]:
now = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

model.verbose=False
hyperparams = {"steps":steps,
              "shift":shift,
              "scale":scale,
              "lr0":lr,
              "mse_scale":mse_scale,
              "mse_shift":mse_shift,
              "batch_size":batch_size}

logdir.mkdir(parents=True, exist_ok=True)
with open(logdir / f"{now}-config.json", "w") as f:
    json.dump(models_config[name], f, indent=4, default=str)
train_config.save(logdir / Path(f"{now}-train_config.json"))
with open(logdir / f"{now}-hyperparams.json", "w") as f:
    json.dump(hyperparams, f, indent=4, default=str)

trainer.fit(lightning_model, train_dataloaders=train_dataloader, val_dataloaders=valid_dataloader)


# Resume from checkpoint

In [ ]:
# Train from checkpoint
trainer.fit(
    lightning_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=valid_dataloader,
    ckpt_path="/media/jlux/SSD2/ReactionDiffusion/fullds/ITLNO_res/20260301-152627/last.ckpt",
)

# Load model

In [ ]:
ckpt_file = Path("/media/jlux/SSD2/ReactionDiffusion/fullds/FNO_res/20260317-110727")
checkpoint = torch.load(ckpt_file/ Path("best_val_loss.ckpt"))
lightning_model.load_state_dict(checkpoint['state_dict'])


# Multi-step model eval

In [ ]:
from utils import plot_interactive, plot_comparison, get_simulation_by_id
# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"

datapath = Path("/media/jlux/SSD2/pdebench/2d_reaction_diffusion/133017.hdf5")
sample_id = train_set.sample_ids[0]
full_inputs = get_simulation_by_id(datapath, sample_id).to("cuda")

print(full_inputs.max().item(), full_inputs.min().item())
full_inputs = (full_inputs + shift) * scale
full_inputs = full_inputs.unsqueeze(0).to(device)

t_norm = 101
pred_list = []
RMSE_list = []

memory = steps
print("memory", memory, "sample id:", sample_id)
print("Full input shape", full_inputs.shape)
print(residual)
start_idx = 5

# Prepare initial input: take first 2 slices and add batch dim
# Shape: [1, 2, C, H, W] (Batch, Sequence, Channels, Height, Width)
current_input = full_inputs[:, start_idx:start_idx + memory]
time_idx = torch.FloatTensor([start_idx + memory - 1])
time_idx = time_idx.unsqueeze(0).to(device)
model = lightning_model.model.to(device)

with torch.no_grad():
    for t in range(start_idx + 1, 100):

        current_time = (time_idx + t + 1) / t_norm

        # Reshape to [Batch, 2*C, H, W] for the model
        B, S, C, H, W = current_input.shape
        model_input = current_input.reshape(B, S * C, H, W)

        # Predict next frame: returns [B, C, H, W]
        preds = model(model_input, time=current_time)
        if not residual:
            preds = full_inputs[:, t, ...] + model.dt * preds
        RMSE_list.append(torch.sqrt(torch.mean((preds - full_inputs[:, t+1, ...]) ** 2)).item())
        print(f"Iteration {t}, error {RMSE_list[-1]}    ", end="\r")
        pred_list.append(preds.squeeze(0).cpu()) # Store on CPU to save VRAM

        # Rolling window: remove oldest, append newest
        # We unsqueeze(1) to get [B, 1, C, H, W] before concat
        current_input = torch.cat([
            current_input[:, 1:, ...],
            preds.unsqueeze(1)
        ], dim=1)

plt.plot(RMSE_list)
plt.yscale('log')
targets = full_inputs[:, start_idx + memory + 1:, ...]
print("Average RMSE:", np.mean(RMSE_list))

# Visualization

In [ ]:
p = torch.stack(pred_list).detach().cpu()
# g = full_inputs[0, start_idx + memory:].detach().cpu()
g = targets[0].detach().cpu()
plot_interactive(p, g, channel_idx=0)

In [ ]:
plt.rc('animation', html='jshtml')
anim = animate_results_mpl(p, g, channel_idx=0)
anim


# Satistics
Compute some statistics on the dataset

In [ ]:
datapath = Path("/media/jlux/SSD2/pdebench/2d_reaction_diffusion/133017.hdf5")
sample_id = train_set.sample_ids[20]
full_inputs = get_simulation_by_id(datapath, sample_id).to("cuda")

print(full_inputs.shape)
print(full_inputs.max().item(), full_inputs.min().item())
print(scale, shift)
dt = 5/100.
full_inputs = (full_inputs + shift) * scale
delta = torch.abs(full_inputs[1:, ...] - full_inputs[:-1, ...]) / dt
delta_mean = torch.mean(delta, dim=(1,2,3))

mean_per_frame = torch.mean(full_inputs, dim=(1,2,3))
std_per_frame = torch.mean(full_inputs**2, dim=(1,2,3)) - mean_per_frame**2
std_total = delta.std()
# print(torch.sqrt(std_per_frame))
print(std_total.item())

delta_max = torch.amax(delta, dim=(1,2,3))
delta_min = torch.amin(delta, dim=(1,2,3))
print(delta_max)
print(delta_min)

vrange = torch.amax(full_inputs, dim=(1,2,3)) - torch.amin(full_inputs,dim=(1,2,3))
plt.plot(torch.sqrt(std_per_frame).cpu().numpy(), label='std')
# plt.plot(mean_per_frame.abs().cpu().numpy(), label='mean')
# plt.plot(vrange.abs().cpu().numpy(), label='range')
plt.plot(delta_mean.cpu().numpy(), label='mean')
plt.plot(delta_min.cpu().numpy(), label='min')
plt.plot(delta_max.cpu().numpy(), label='max')
plt.yscale("log")
plt.legend()
print(vrange.mean().item(), std_per_frame.mean().item(), mean_per_frame.mean().item())

# END